<a href="https://colab.research.google.com/github/choudhary-hema/Assignment-3-100-Next-Gen-AI-Engineers-Training-Cohort-5-/blob/main/Gemini_RAG_ML_Knowledge_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini RAG Knowledge Assistant
### UST Assignment 6 — Retrieval-Augmented Generation (RAG)


## 1. Install and import dependencies

In [ ]:
%pip -q install -U "google-genai<3" pymupdf faiss-cpu "numpy<2.5" "pandas==2.2.3" "google-auth==2.49.0" matplotlib langchain-text-splitters

In [ ]:
import os
import re
import time
import getpass
from pathlib import Path

import fitz  # PyMuPDF
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imports completed successfully.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Imports completed successfully.


## 2. Configure Gemini



In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key (input hidden): ")

if not GEMINI_API_KEY:
    raise ValueError("A Gemini API key is required.")

client = genai.Client(api_key=GEMINI_API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.6-flash"

print("Gemini client configured.")
print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)

Enter your Gemini API key (input hidden): ··········
Gemini client configured.
Embedding model: gemini-embedding-001
Generation model: gemini-3.6-flash


## 3. Load the troubleshooting PDF

In [ ]:
PDF_PATH = Path("/content/UST - Assignment 3 - RAG - ML Knowledge Assistant.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Could not find PDF: {PDF_PATH.resolve()}")

print(f"Using PDF: {PDF_PATH.resolve()}")

Using PDF: /content/UST - Assignment 3 - RAG - ML Knowledge Assistant.pdf


## 4. Extract text and inspect document structure

In [ ]:
doc = fitz.open(PDF_PATH)

pages = []
for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text("text")
    # Normalize whitespace while preserving paragraph boundaries reasonably well.
    page_text = page_text.replace("\\x00", " ")
    page_text = re.sub(r"[ \\t]+", " ", page_text)
    page_text = re.sub(r"\\n{3,}", "\\n\\n", page_text).strip()
    pages.append({
        "page": page_number,
        "text": page_text,
        "characters": len(page_text)
    })

document_stats = pd.DataFrame([
    {"Page": p["page"], "Extracted characters": p["characters"]}
    for p in pages
])

print(f"Number of pages: {len(pages)}")
print(f"Total extracted characters: {sum(p['characters'] for p in pages):,}")
print("Pages with fewer than 50 extracted characters:",
      [p["page"] for p in pages if p["characters"] < 50])
display(document_stats)

print("\\nPreview (first page):\\n")
print(pages[0]["text"][:2000])

Number of pages: 7
Total extracted characters: 16,144
Pages with fewer than 50 extracted characters: []


,Page,Extracted characters
0,1,2221
1,2,1841
2,3,2200
3,4,2576
4,5,2630
5,6,2617
6,7,2059


\nPreview (first page):\n
Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a
Page 1
Con oso Pla form Troubleshoo ing Runbook
Documen owner: Pla form Reliabili y Team | Version: 3.2 | Las reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consis en way o diagnose and resolve produc ion inciden s
across he eigh core Con oso services: paymen -ga eway, au h-service, order-api, inven ory-service,
no ifica ion-service, search-service, repor ing-engine and user-profile-api. I covers he en mos frequen 
error codes seen in he inciden log over he las wo quar ers. Each error code has a symp om descrip ion,
likely causes, a diagnos ic checklis , a fix, and a preven ion no e.
2. How o use his runbook
S ar from he error code in he aler or log line. Confirm he symp om ma ches, work hrough he
diagnos ic checklis in order, and apply he fix only af er he cause is confirmed. If wo consecu ive
diagnos ic s eps produce no signal, escal

### 4.1 Text quality checks

PDF extraction can produce broken line wraps, headers/footers, unusual symbols or missing text. Review the preview and page character counts. This runbook contains text and tables; if a page is image-only or extraction is incomplete, OCR would be a separate preprocessing enhancement.

In [ ]:
empty_pages = [p["page"] for p in pages if not p["text"].strip()]
print("Empty pages:", empty_pages if empty_pages else "None")
print("Average characters per page:", round(np.mean([p["characters"] for p in pages]), 1))

Empty pages: None
Average characters per page: 2306.3


## 5. Split text into overlapping chunks

In [ ]:
CHUNK_SIZE = 3500
CHUNK_OVERLAP = 500

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\\n\\n", "\\n", ". ", " ", ""],
    length_function=len,
)

chunks = []
for page in pages:
    if not page["text"]:
        continue
    page_chunks = text_splitter.split_text(page["text"])
    for local_index, chunk_text in enumerate(page_chunks):
        chunks.append({
            "chunk_id": len(chunks),
            "page": page["page"],
            "text": chunk_text,
            "characters": len(chunk_text)
        })

chunks_df = pd.DataFrame(chunks)
print(f"Total chunks created: {len(chunks)}")
display(chunks_df[["chunk_id", "page", "characters"]].head(10))
print("\\nExample chunk:\\n")
print(chunks[0]["text"][:1500])

Total chunks created: 7


,chunk_id,page,characters
0,0,1,2221
1,1,2,1841
2,2,3,2200
3,3,4,2576
4,4,5,2630
5,5,6,2617
6,6,7,2059


\nExample chunk:\n
Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a
Page 1
Con oso Pla form Troubleshoo ing Runbook
Documen owner: Pla form Reliabili y Team | Version: 3.2 | Las reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consis en way o diagnose and resolve produc ion inciden s
across he eigh core Con oso services: paymen -ga eway, au h-service, order-api, inven ory-service,
no ifica ion-service, search-service, repor ing-engine and user-profile-api. I covers he en mos frequen 
error codes seen in he inciden log over he las wo quar ers. Each error code has a symp om descrip ion,
likely causes, a diagnos ic checklis , a fix, and a preven ion no e.
2. How o use his runbook
S ar from he error code in he aler or log line. Confirm he symp om ma ches, work hrough he
diagnos ic checklis in order, and apply he fix only af er he cause is confirmed. If wo consecu ive
diagnos ic s eps produce no signal, escala e acc

## 6. Generate Gemini embeddings

In [ ]:
def embed_texts(texts, task_type, batch_size=16, pause_seconds=1.0):
    """Create Gemini embeddings for a list of strings, preserving input order."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        last_error = None
        for attempt in range(4):
            try:
                response = client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type)
                )
                batch_vectors = [np.asarray(item.values, dtype=np.float32)
                                 for item in response.embeddings]
                if len(batch_vectors) != len(batch):
                    raise RuntimeError("Embedding response count did not match input count.")
                vectors.extend(batch_vectors)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                if attempt == 3:
                    raise
                wait = 2 ** attempt
                print(f"Embedding batch failed ({exc}); retrying in {wait}s...")
                time.sleep(wait)
        if last_error is not None:
            raise last_error
        time.sleep(pause_seconds)
    return np.vstack(vectors).astype(np.float32)

chunk_texts = [chunk["text"] for chunk in chunks]
document_embeddings = embed_texts(
    chunk_texts,
    task_type="RETRIEVAL_DOCUMENT",
    batch_size=16
)

print("Embedding matrix shape:", document_embeddings.shape)
print("Number of vectors:", len(document_embeddings))
print("Vector dimensions:", document_embeddings.shape[1])

Embedding matrix shape: (7, 3072)
Number of vectors: 7
Vector dimensions: 3072


## 7. Store embeddings in FAISS

In [ ]:
faiss.normalize_L2(document_embeddings)

embedding_dimension = document_embeddings.shape[1]
vector_index = faiss.IndexFlatIP(embedding_dimension)
vector_index.add(document_embeddings)

print("FAISS index created.")
print("Vectors stored:", vector_index.ntotal)
print("Embedding dimension:", embedding_dimension)

FAISS index created.
Vectors stored: 7
Embedding dimension: 3072


## 8. Implement query-based retrieval

In [ ]:
def retrieve(query, top_k=5):
    query_response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_vector = np.asarray(
        query_response.embeddings[0].values, dtype=np.float32
    ).reshape(1, -1)
    faiss.normalize_L2(query_vector)

    scores, indices = vector_index.search(query_vector, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = dict(chunks[int(idx)])
        item["similarity"] = float(score)
        results.append(item)
    return results

sample_query = "What causes DB-POOL errors and how should an engineer resolve them?"
sample_results = retrieve(sample_query, top_k=5)

display(pd.DataFrame([
    {
        "Rank": rank,
        "Chunk ID": item["chunk_id"],
        "Source page": item["page"],
        "Cosine similarity": round(item["similarity"], 4),
        "Preview": item["text"][:220].replace("\\n", " ")
    }
    for rank, item in enumerate(sample_results, start=1)
]))

,Rank,Chunk ID,Source page,Cosine similarity,Preview
0,1,4,5,0.7594,Con oso Pla form Engineering | Troubleshoo ing...
1,2,5,6,0.7086,Con oso Pla form Engineering | Troubleshoo ing...
2,3,3,4,0.7049,Con oso Pla form Engineering | Troubleshoo ing...
3,4,2,3,0.6981,Con oso Pla form Engineering | Troubleshoo ing...
4,5,6,7,0.6905,Con oso Pla form Engineering | Troubleshoo ing...


## 9. Experiment with different Top-K values

In [ ]:
top_k_experiment_query = "How do I troubleshoot an expired authentication token (ERR-401)?"
top_k_results = {}

for k in [3, 5, 7]:
    results = retrieve(top_k_experiment_query, top_k=k)
    top_k_results[k] = results
    print(f"\\nTop-K = {k}")
    for rank, item in enumerate(results, start=1):
        print(
            f"{rank}. Page {item['page']} | "
            f"score={item['similarity']:.4f} | "
            f"{item['text'][:180].replace(chr(10), ' ')}..."
        )

\nTop-K = 3
1. Page 3 | score=0.7470 | Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a Page 3 Au hen ica ion and Ra e Limi ing Errors ERR-401 - Expired or invalid au hen ica ...
2. Page 6 | score=0.6534 | Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a Page 6 Infras ruc ure Errors MEM-OOM - Con ainer OOMKilled Symp om: Pods res ar repea e...
3. Page 4 | score=0.6529 | Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a Page 4 Applica ion and Ga eway Errors ERR-500 - Unhandled excep ion Symp om: HTTP 500 w...
\nTop-K = 5
1. Page 3 | score=0.7470 | Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a Page 3 Au hen ica ion and Ra e Limi ing Errors ERR-401 - Expired or invalid au hen ica ...
2. Page 6 | score=0.6534 | Con oso Pla form Engineering | Troubleshoo ing Runbook v3.2 | In ernal - Syn he ic demo da a Page 6 Infra

## 10. Build a grounded Gemini answer-generation prompt

In [ ]:
SYSTEM_INSTRUCTION = """
You are a technical knowledge assistant for the Contoso Platform Troubleshooting Runbook.
Answer the user's question using ONLY the supplied retrieved context.
Do not invent commands, causes, thresholds, owners, or remediation steps.
If the context does not contain enough information, clearly say:
"The supplied runbook context does not provide enough information to answer that."
Explain the relevant diagnostic or remediation steps in a clear, practical order.
Cite evidence using the source page labels, for example [Source: page 5].
If multiple pages support the answer, cite each relevant page.
""".strip()


def generate_rag_answer(question, top_k=5, temperature=0.2):
    retrieved = retrieve(question, top_k=top_k)

    context_parts = []
    for item in retrieved:
        context_parts.append(
            f"[Source: page {item['page']}; chunk {item['chunk_id']}]\n"
            f"{item['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
Retrieved context:
{context}

User question:
{question}

Write a concise, evidence-grounded answer. Include page citations.
""".strip()

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=temperature,
            max_output_tokens=900
        )
    )

    return {
        "question": question,
        "answer": response.text or "No text response was returned.",
        "retrieved": retrieved
    }

## 11. Run an end-to-end question-answering test

In [ ]:
question = "What causes DB-POOL errors, what diagnostics should I perform, and what is the recommended fix?"
rag_result = generate_rag_answer(question, top_k=5)

print("QUESTION:")
print(rag_result["question"])
print("\\nGEMINI ANSWER:")
print(rag_result["answer"])

print("\\nRETRIEVED EVIDENCE:")
for rank, item in enumerate(rag_result["retrieved"], start=1):
    print(f"\\n[{rank}] Page {item['page']} | Similarity {item['similarity']:.4f}")
    print(item["text"][:1200])

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 12. Test multiple troubleshooting scenarios

In [ ]:
test_questions = [
    "What are the likely causes and fixes for ERR-401?",
    "How should clients handle ERR-429 rate limiting?",
    "What should an engineer check first when ERR-500 starts after a deployment?",
    "What should be done when a downstream dependency causes ERR-503?",
    "How do I diagnose and resolve DB-POOL connection pool exhaustion?",
    "What does OOMKilled with exit code 137 indicate, and what are the recommended actions?",
    "How should the team respond to a TLS certificate expiration (CERT-EXP)?",
    "What are the acknowledgement targets and update cadences for Sev1 and Sev2?"
]

evaluation_outputs = []
for q in test_questions:
    print("\\n" + "=" * 90)
    print("QUESTION:", q)
    result = generate_rag_answer(q, top_k=5)
    print("\\nANSWER:\\n", result["answer"])
    source_pages = sorted(set(item["page"] for item in result["retrieved"]))
    evaluation_outputs.append({
        "Question": q,
        "Retrieved source pages": ", ".join(map(str, source_pages)),
        "Answer": result["answer"]
    })

evaluation_df = pd.DataFrame(evaluation_outputs)

## 13. Basic retrieval evaluation and observations

In [ ]:
display(evaluation_df[["Question", "Retrieved source pages"]])

### 13.1 Test an out-of-scope question

A grounded RAG assistant should not fabricate an answer when the source does not cover the topic. This test asks about a topic not described in the runbook.

In [ ]:
out_of_scope_question = "What is the process for configuring a Kubernetes cluster autoscaler from scratch?"
out_of_scope_result = generate_rag_answer(out_of_scope_question, top_k=5)

print("QUESTION:", out_of_scope_question)
print("\\nANSWER:")
print(out_of_scope_result["answer"])

In [ ]:
import pickle

VECTOR_DIR = Path("rag_artifacts")
VECTOR_DIR.mkdir(exist_ok=True)

faiss.write_index(vector_index, str(VECTOR_DIR / "runbook_faiss.index"))
with open(VECTOR_DIR / "runbook_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Saved FAISS index and chunk metadata to:", VECTOR_DIR.resolve())

Saved FAISS index and chunk metadata to: /content/rag_artifacts
